In [2]:
import numpy as np
import random
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import autosklearn.classification

In [ ]:
search_space = {
    "logistic_regression": {
        "hyperparams": {
            "C": [0.0001, 0.001, 0.01, 0.1, 1.0, 10, 100],
            "solver": ["liblinear", "lbfgs"]
        },
        "build_fn": lambda hp: LogisticRegression(
            C=hp["C"], 
            solver=hp["solver"],
            max_iter=10000
        )
    },
    "random_forest": {
        "hyperparams": {
            "n_estimators": [5, 10, 20, 30, 50, 100, 200],
            "max_depth": [None, 5, 10, 15, 20, 25, 30],
            "criterion": ["gini", "entropy"]
        },
        "build_fn": lambda hp: RandomForestClassifier(
            n_estimators=hp["n_estimators"],
            max_depth=hp["max_depth"],
            criterion=hp["criterion"],
            random_state=42
        )
    },
    "gradient_boosting": {
        "hyperparams": {
            "n_estimators": [5, 10, 20, 30, 50, 100, 200],
            "learning_rate": [0.00001, 0.0001, 0.001, 0.01, 0.1],
            "max_depth": [3, 5, 7, 9]
        },
        "build_fn": lambda hp: GradientBoostingClassifier(
            n_estimators=hp["n_estimators"],
            learning_rate=hp["learning_rate"],
            max_depth=hp["max_depth"],
            random_state=42
        )
    },
    "svc": {
        "hyperparams": {
            "C": [0.01, 0.1, 1.0, 10],
            "kernel": ["linear", "rbf"],
            "gamma": ["scale", "auto"]
        },
        "build_fn": lambda hp: SVC(
            C=hp["C"],
            kernel=hp["kernel"],
            gamma=hp["gamma"],
            probability=True, # so we can get predict_proba if needed
            random_state=42
        )
    },
    "knn": {
        "hyperparams": {
            "n_neighbors": [1, 3, 5, 7, 9],
            "weights": ["uniform", "distance"],
            "p": [1, 2]  # 1 => Manhattan distance, 2 => Euclidean distance
        },
        "build_fn": lambda hp: KNeighborsClassifier(
            n_neighbors=hp["n_neighbors"],
            weights=hp["weights"],
            p=hp["p"]
        )
    }
}

In [34]:
def random_solution(search_space):
    """
    Generate a random solution from the defined search space.
    A solution is a dict:
    {
      "algo_name": <string>,
      "hyperparams": <dict with each param chosen randomly from the possible range>
    }
    """
    algo_name = random.choice(list(search_space.keys()))
    hyperparams_choices = search_space[algo_name]["hyperparams"]

    chosen_hyperparams = {}
    for param_name, possible_values in hyperparams_choices.items():
        chosen_hyperparams[param_name] = random.choice(possible_values)
    
    return {
        "algo_name": algo_name,
        "hyperparams": chosen_hyperparams
    }


def get_neighbor(solution, search_space):
    """
    Given the current solution, produce a 'neighbor' solution by randomly
    changing either the algorithm or one of the hyperparameters.
    We define the 'neighbor' as:
      - with 50% chance, switch the algorithm to a different one entirely
      - otherwise, pick one hyperparameter and change it to a different 
        permissible value in that hyperparam's range.
    """
    neighbor_sol = {
        "algo_name": solution["algo_name"],
        "hyperparams": solution["hyperparams"].copy()
    }
    
    # Decide whether we switch algorithm or tune a hyperparam
    if random.random() < 0.50:
        # Switch algorithm
        new_algo = random.choice(list(search_space.keys()))
        while new_algo == neighbor_sol["algo_name"]:
            new_algo = random.choice(list(search_space.keys()))
        # pick random hyperparams for the new algorithm
        hyperparams_choices = search_space[new_algo]["hyperparams"]
        chosen_hyperparams = {}
        for param_name, possible_values in hyperparams_choices.items():
            chosen_hyperparams[param_name] = random.choice(possible_values)
        
        neighbor_sol["algo_name"] = new_algo
        neighbor_sol["hyperparams"] = chosen_hyperparams
    else:
        # Switch one hyperparam from the same algorithm
        algo_name = neighbor_sol["algo_name"]
        hyperparams_choices = search_space[algo_name]["hyperparams"]
        param_to_change = random.choice(list(hyperparams_choices.keys()))
        
        current_value = neighbor_sol["hyperparams"][param_to_change]
        possible_values = hyperparams_choices[param_to_change]
        
        # pick a new value for that param (different from the current)
        new_value = random.choice(possible_values)
        while new_value == current_value and len(possible_values) > 1:
            new_value = random.choice(possible_values)
        
        neighbor_sol["hyperparams"][param_to_change] = new_value
    
    return neighbor_sol


def build_model(solution, search_space):
    """
    Given a solution (algorithm + hyperparams),
    build and return the corresponding sklearn model.
    """
    algo_name = solution["algo_name"]
    hp = solution["hyperparams"]
    return search_space[algo_name]["build_fn"](hp)


def evaluate_solution(solution, X, y, cv_folds=5):
    """
    Evaluate a solution by training/testing via cross-validation.
    The evaluation metric is classification accuracy (higher is better).
    
    Returns the negative accuracy for minimization purposes 
    (i.e., we want to *minimize* negative accuracy, i.e. maximize accuracy).
    If you prefer other metrics or a regression setting, adapt accordingly.
    """
    model = build_model(solution, search_space)
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    mean_score = np.mean(scores)
    
    # We transform the objective so that "lower is better" --> negative accuracy
    # Alternatively, we can keep it as 1 - accuracy, or any cost function you prefer.
    cost = -mean_score
    return cost



In [35]:
def simulated_annealing(
    X, y,
    search_space,
    max_iterations=50,
    initial_temperature=1.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
):
    """
    Perform Simulated Annealing to find best (algorithm, hyperparameters)
    that maximizes accuracy (or equivalently minimizes negative accuracy).

    Parameters:
    -----------
    X, y:       Training data (features, labels)
    search_space: dict describing algorithms and hyperparams
    max_iterations: how many outer iterations (cooling steps)
    initial_temperature: starting "temperature"
    min_temperature: minimal temperature to stop
    alpha: cooling ratio T <- alpha * T
    inner_loop: how many neighbor solutions to try at each temperature
    random_seed: for reproducibility

    Returns:
    --------
    best_solution, best_cost
    """
    random.seed(random_seed)
    np.random.seed(random_seed)

    # Initialize
    current_solution = random_solution(search_space)
    current_cost = evaluate_solution(current_solution, X, y)
    best_solution = current_solution
    best_cost = current_cost

    T = initial_temperature
    iteration = 0

    # Start SA loop
    while T > min_temperature and iteration < max_iterations:
        print(f"Iteration {iteration + 1}/{max_iterations} - Temperature: {T:.4f} - Current Cost: {current_cost:.4f} - Best Cost: {best_cost:.4f}")
        for inner_iter in range(inner_loop):
            # get neighbor
            neighbor = get_neighbor(current_solution, search_space)
            neighbor_cost = evaluate_solution(neighbor, X, y)

            # if neighbor is better, accept it
            if neighbor_cost < current_cost:
                current_solution = neighbor
                current_cost = neighbor_cost
                print(f"  Inner {inner_iter + 1}/{inner_loop}: Accepted better solution with cost: {current_cost:.4f}")
            else:
                # accept with probability e^(-(neighbor_cost - current_cost)/T)
                cost_diff = neighbor_cost - current_cost
                acceptance_prob = np.exp(-cost_diff / T)
                if random.random() < acceptance_prob:
                    current_solution = neighbor
                    current_cost = neighbor_cost
                    print(f"  Inner {inner_iter + 1}/{inner_loop}: Accepted worse solution with cost: {current_cost:.4f} (prob: {acceptance_prob:.4f})")

            # update global best if needed
            if current_cost < best_cost:
                best_solution = current_solution
                best_cost = current_cost
                print(f"  New Best Found: Cost: {best_cost:.4f}, Algorithm: {best_solution['algo_name']}")

        # cool down
        T = alpha * T
        iteration += 1

    return best_solution, best_cost



# Load a sample classification dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Run Simulated Annealing
start_time = time.time()
best_solution, best_cost = simulated_annealing(
    X, y,
    search_space=search_space,
    max_iterations=40,         # you can tune these parameters
    initial_temperature=2.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
)
runtime = time.time() - start_time

# best_cost is negative accuracy (i.e., -accuracy)
best_accuracy = -best_cost

print("Simulated Annealing best solution found:")
print("  Algorithm:", best_solution["algo_name"])
print("  Hyperparameters:", best_solution["hyperparams"])
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  Total runtime: {runtime:.2f} seconds")


Iteration 1/40 - Temperature: 2.0000 - Current Cost: -0.9280 - Best Cost: -0.9280
  Inner 1/10: Accepted better solution with cost: -0.9561
  New Best Found: Cost: -0.9561, Algorithm: random_forest
  Inner 2/10: Accepted worse solution with cost: -0.9456 (prob: 0.9947)
  Inner 3/10: Accepted worse solution with cost: -0.9139 (prob: 0.9843)
  Inner 4/10: Accepted better solution with cost: -0.9280
  Inner 5/10: Accepted better solution with cost: -0.9315


/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode, _ = stats.mode(_y[neigh_ind, k], axis=1)
/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to Tr

  Inner 6/10: Accepted worse solution with cost: -0.6274 (prob: 0.8590)
  Inner 7/10: Accepted worse solution with cost: -0.6274 (prob: 1.0000)
  Inner 8/10: Accepted worse solution with cost: -0.6274 (prob: 1.0000)
  Inner 9/10: Accepted better solution with cost: -0.9385
  Inner 10/10: Accepted worse solution with cost: -0.9368 (prob: 0.9991)
Iteration 2/40 - Temperature: 1.7000 - Current Cost: -0.9368 - Best Cost: -0.9561
  Inner 1/10: Accepted worse solution with cost: -0.9245 (prob: 0.9928)
  Inner 2/10: Accepted worse solution with cost: -0.6274 (prob: 0.8397)
  Inner 3/10: Accepted better solution with cost: -0.9508
  Inner 4/10: Accepted worse solution with cost: -0.9262 (prob: 0.9856)
  Inner 5/10: Accepted better solution with cost: -0.9368
  Inner 6/10: Accepted better solution with cost: -0.9613
  New Best Found: Cost: -0.9613, Algorithm: logistic_regression
  Inner 7/10: Accepted better solution with cost: -0.9631
  New Best Found: Cost: -0.9631, Algorithm: random_forest
 

/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode, _ = stats.mode(_y[neigh_ind, k], axis=1)
/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to Tr

  Inner 1/10: Accepted better solution with cost: -0.9403
  Inner 2/10: Accepted better solution with cost: -0.9561
  Inner 3/10: Accepted worse solution with cost: -0.9561 (prob: 1.0000)
  Inner 4/10: Accepted worse solution with cost: -0.6274 (prob: 0.7966)
  Inner 5/10: Accepted worse solution with cost: -0.6274 (prob: 1.0000)
  Inner 6/10: Accepted better solution with cost: -0.7944
  Inner 7/10: Accepted better solution with cost: -0.9139
  Inner 8/10: Accepted worse solution with cost: -0.6274 (prob: 0.8202)
  Inner 9/10: Accepted better solution with cost: -0.9139
  Inner 10/10: Accepted better solution with cost: -0.9473
Iteration 4/40 - Temperature: 1.2282 - Current Cost: -0.9473 - Best Cost: -0.9631
  Inner 1/10: Accepted worse solution with cost: -0.6274 (prob: 0.7707)
  Inner 2/10: Accepted worse solution with cost: -0.6274 (prob: 1.0000)
  Inner 3/10: Accepted worse solution with cost: -0.6274 (prob: 1.0000)
  Inner 4/10: Accepted better solution with cost: -0.9385
  Inner

...

  Inner 10/10: Accepted better solution with cost: -0.9596
Iteration 37/40 - Temperature: 0.0058 - Current Cost: -0.9596 - Best Cost: -0.9631
  Inner 3/10: Accepted worse solution with cost: -0.9596 (prob: 1.0000)
  Inner 5/10: Accepted worse solution with cost: -0.9596 (prob: 1.0000)
  Inner 6/10: Accepted worse solution with cost: -0.9596 (prob: 1.0000)
Iteration 38/40 - Temperature: 0.0049 - Current Cost: -0.9596 - Best Cost: -0.9631
  Inner 1/10: Accepted worse solution with cost: -0.9561 (prob: 0.4866)
  Inner 4/10: Accepted better solution with cost: -0.9578
  Inner 6/10: Accepted worse solution with cost: -0.9578 (prob: 1.0000)
  Inner 10/10: Accepted worse solution with cost: -0.9578 (prob: 1.0000)
Iteration 39/40 - Temperature: 0.0042 - Current Cost: -0.9578 - Best Cost: -0.9631


/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode, _ = stats.mode(_y[neigh_ind, k], axis=1)
/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to Tr

  Inner 7/10: Accepted worse solution with cost: -0.9578 (prob: 1.0000)
  Inner 8/10: Accepted worse solution with cost: -0.9578 (prob: 1.0000)
Iteration 40/40 - Temperature: 0.0035 - Current Cost: -0.9578 - Best Cost: -0.9631


/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode, _ = stats.mode(_y[neigh_ind, k], axis=1)
/usr/local/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:211: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to Tr

  Inner 5/10: Accepted worse solution with cost: -0.9578 (prob: 1.0000)
  Inner 6/10: Accepted better solution with cost: -0.9596
  Inner 9/10: Accepted worse solution with cost: -0.9596 (prob: 1.0000)
Simulated Annealing best solution found:
  Algorithm: random_forest
  Hyperparameters: {'n_estimators': 50, 'max_depth': 25, 'criterion': 'entropy'}
  Accuracy: 0.9631
  Total runtime: 398.24 seconds


In [36]:
automl = autosklearn.classification.AutoSklearnClassifier(time_left_for_this_task=60, seed=42)
automl.fit(X, y)
print("Auto-Sklearn Best Score:", automl.score(X, y))

y_pred_automl = automl.predict(X)

accuracy_automl = automl.score(X, y)

# Extract the best configuration
best_model_automl = automl.show_models()
best_config_automl = automl.get_models_with_weights()

# Print Auto-Sklearn Results
print("Auto-Sklearn Best Solution Found:")
print(f"  Best Configuration: {best_config_automl}")
print(f"  Accuracy: {accuracy_automl:.4f}")

Auto-Sklearn Best Score: 0.9876977152899824
Auto-Sklearn Best Solution Found:
  Best Configuration: [(0.2, SimpleClassificationPipeline({'balancing:strategy': 'none', 'classifier:__choice__': 'extra_trees', 'data_preprocessor:__choice__': 'feature_type', 'feature_preprocessor:__choice__': 'polynomial', 'classifier:extra_trees:bootstrap': 'False', 'classifier:extra_trees:criterion': 'gini', 'classifier:extra_trees:max_depth': 'None', 'classifier:extra_trees:max_features': 0.5707983257382487, 'classifier:extra_trees:max_leaf_nodes': 'None', 'classifier:extra_trees:min_impurity_decrease': 0.0, 'classifier:extra_trees:min_samples_leaf': 3, 'classifier:extra_trees:min_samples_split': 11, 'classifier:extra_trees:min_weight_fraction_leaf': 0.0, 'data_preprocessor:feature_type:numerical_transformer:imputation:strategy': 'median', 'data_preprocessor:feature_type:numerical_transformer:rescaling:__choice__': 'none', 'feature_preprocessor:polynomial:degree': 2, 'feature_preprocessor:polynomial:inc